In [2]:
# =============================================================================
# CELL 1 — Install Dependencies
# =============================================================================
# Purpose : Install ENTSO-E Python client for day-ahead price fetching.
#           Runs once per session — cached by Fabric after first install.
# =============================================================================

%pip install entsoe-py requests --quiet

print("✅ Dependencies installed")

StatementMeta(, 4f4dfe2d-dd38-4fee-8fb2-abc63e948b5f, 15, Finished, Available, Finished, False)


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ Dependencies installed



In [3]:
# =============================================================================
# CELL 2 — Imports & Configuration
# =============================================================================
# Purpose : Load API credentials from Fabric Environment (pulsegrid_env)
#           and define all configuration for the daily price poller.
#
# Schedule : Runs once per day at 13:00 CET via Fabric Pipeline
#            ENTSO-E day-ahead prices are published daily at ~12:00 CET
#            We poll at 13:00 to ensure publication is complete
#
# Regions  : 26 European bidding zones
#            Covers all major EU electricity markets
# =============================================================================

import time
import random
import pandas as pd
from datetime import datetime, timezone, timedelta
from entsoe import EntsoePandasClient
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, TimestampType
)
from delta.tables import DeltaTable

# -----------------------------------------------------------------------------
# API Credentials — from Fabric Environment
# -----------------------------------------------------------------------------
ENTSOE_API_TOKEN = spark.conf.get("spark.pulsegrid.entsoe_token")
print(f"✅ ENTSO-E token loaded: {ENTSOE_API_TOKEN[:8]}...{ENTSOE_API_TOKEN[-4:]}")

# -----------------------------------------------------------------------------
# Rate Limiting
# -----------------------------------------------------------------------------
MAX_RETRIES    = 3
BACKOFF_FACTOR = 2
JITTER_MAX     = 1.5

# -----------------------------------------------------------------------------
# KQL Bronze Target
# -----------------------------------------------------------------------------
KUSTO_CLUSTER  = "https://trd-ratdj1p1b0yurnmn41.z4.kusto.fabric.microsoft.com"
KUSTO_DATABASE = "pulsegrid_bronze"
KUSTO_TABLE    = "raw_electricity_prices"

# -----------------------------------------------------------------------------
# ENTSO-E Bidding Zones — 26 European regions
# -----------------------------------------------------------------------------
ENTSOE_REGIONS = {
    # Working zones — keep as is
    "FR": "10YFR-RTE------C",
    "ES": "10YES-REE------0",
    "NL": "10YNL----------L",
    "BE": "10YBE----------2",
    "PL": "10YPL-AREA-----S",
    "AT": "10YAT-APG------L",
    "CH": "10YCH-SWISSGRIDZ",
    "PT": "10YPT-REN------W",
    "FI": "10YFI-1--------U",
    "CZ": "10YCZ-CEPS-----N",
    "SK": "10YSK-SEPS-----K",
    "HU": "10YHU-MAVIR----U",
    "RO": "10YRO-TEL------P",
    "BG": "10YCA-BULGARIA-R",
    "HR": "10YHR-HEP------M",
    "GR": "10YGR-HTSO-----Y",
    "SI": "10YSI-ELES-----O",
    "RS": "10YCS-SERBIATSOV",
    "LT": "10YLT-1001A0008Q",
    "LV": "10YLV-1001A00074",

    # Fixed sub-zone keys for previously failing zones
    "DE-LU": "10Y1001A1001A82H",  # Germany-Luxembourg (correct DE key)
    "IT-NO": "10Y1001A1001A73I",  # Italy North
    "DK-1" : "10YDK-1--------W",  # Denmark West
    "DK-2" : "10YDK-2--------M",  # Denmark East
    "SE-3" : "10Y1001A1001A46L",  # Sweden zone 3 (largest)
    "NO-2" : "10YNO-2--------T",  # Norway zone 2
    "EE"   : "10Y1001A1001A39I",  # Estonia
}

print(f"✅ Config loaded")
print(f"   Regions to poll : {len(ENTSOE_REGIONS)}")
print(f"   Target table    : {KUSTO_DATABASE}.{KUSTO_TABLE}")

StatementMeta(, 4f4dfe2d-dd38-4fee-8fb2-abc63e948b5f, 17, Finished, Available, Finished, False)

✅ ENTSO-E token loaded: f3e904f3...6b27
✅ Config loaded
   Regions to poll : 27
   Target table    : pulsegrid_bronze.raw_electricity_prices


In [4]:
# =============================================================================
# CELL 3 — Retry Helper with Exponential Backoff + Jitter
# =============================================================================
# Purpose : Wrap API calls with retry logic to handle transient failures.
#
# Exponential backoff:
#   Attempt 1 fail → wait 2^1 + jitter ≈ 2-3s
#   Attempt 2 fail → wait 2^2 + jitter ≈ 4-5s
#   Attempt 3 fail → wait 2^3 + jitter ≈ 8-9s
#
# Jitter : Prevents thundering herd if pipeline triggers multiple instances
# =============================================================================

def call_with_retry(fn, *args, **kwargs):
    last_exception = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_exception = e
            wait = (BACKOFF_FACTOR ** attempt) + random.uniform(0, JITTER_MAX)
            print(f"   ⚠️  Attempt {attempt}/{MAX_RETRIES} failed: {type(e).__name__}: {e}")
            print(f"   Retrying in {wait:.1f}s...")
            time.sleep(wait)
    print(f"   ❌ All {MAX_RETRIES} attempts failed")
    raise last_exception

print("✅ Retry helper defined")

StatementMeta(, 4f4dfe2d-dd38-4fee-8fb2-abc63e948b5f, 18, Finished, Available, Finished, False)

✅ Retry helper defined


In [5]:
# =============================================================================
# CELL 4 — ENTSO-E Day-Ahead Price Fetcher
# =============================================================================
# Purpose : Fetch day-ahead electricity prices for all 26 bidding zones
#           for today's delivery date.
#
# API Details:
#   - query_day_ahead_prices returns hourly prices for the delivery day
#   - Published once/day at ~12:00 CET for the next day
#   - We fetch today's prices — 24 hourly records per zone
#   - 26 zones × 24 hours = 624 records per daily run
#
# Rate limit math:
#   - 26 API calls total per daily run
#   - 0.02 calls/min average — negligible vs 400/min limit
#   - Inter-request delay: 1s + jitter to be conservative
# =============================================================================

def fetch_day_ahead_prices(region_code, zone_key, client, delivery_date):
    """Fetch day-ahead prices for one bidding zone."""

    start = pd.Timestamp(
        delivery_date.year, delivery_date.month, delivery_date.day,
        tz="Europe/Brussels"
    )
    end = start + pd.Timedelta(days=1)

    prices = client.query_day_ahead_prices(zone_key, start=start, end=end)

    records = []
    ingestion_time = datetime.now(timezone.utc)

    for ts, price in prices.items():
        records.append({
            "ingestion_time": ingestion_time,
            "event_time"    : ts.to_pydatetime(),
            "region"        : region_code,
            "price_eur_mwh" : float(price),
            "load_mw"       : None,
            "temperature_c" : None,
            "source"        : "ENTSO-E"
        })

    return records


def fetch_all_day_ahead_prices():
    """Fetch day-ahead prices for all 26 zones."""

    client        = EntsoePandasClient(api_key=ENTSOE_API_TOKEN)
    delivery_date = datetime.now(timezone.utc).date()
    all_records   = []
    failed        = []

    print(f"\n📡 Fetching day-ahead prices for {delivery_date}")
    print(f"   Zones to poll: {len(ENTSOE_REGIONS)}")

    for region_code, zone_key in ENTSOE_REGIONS.items():
        try:
            records = call_with_retry(
                fetch_day_ahead_prices,
                region_code, zone_key, client, delivery_date
            )
            all_records.extend(records)
            print(f"   ✅ {region_code}: {len(records)} hourly records")

            # Conservative inter-request delay
            time.sleep(1 + random.uniform(0, 0.5))

        except Exception as e:
            failed.append(region_code)
            print(f"   ❌ {region_code}: failed — {e}")

    print(f"\n   Total records  : {len(all_records)}")
    print(f"   Failed regions : {failed if failed else 'None'}")
    return all_records

print("✅ Day-ahead price fetcher defined")

StatementMeta(, 4f4dfe2d-dd38-4fee-8fb2-abc63e948b5f, 19, Finished, Available, Finished, False)

✅ Day-ahead price fetcher defined


In [6]:
# =============================================================================
# CELL 5 — Write Records to Bronze KQL Table (Fixed)
# =============================================================================
# Purpose : Convert fetched records to Spark DataFrame and append to
#           raw_electricity_prices in pulsegrid_bronze.
#
# Fix — Arrow serialization error:
#   - Arrow optimization fails when pandas DataFrame has None values
#     mixed with numeric types (BufferHolder negative size error)
#   - Solution 1: Disable Arrow for this specific operation
#   - Solution 2: Explicitly cast all nullable columns before createDataFrame
#   - Both applied here for robustness
#
# Write mode: Append only — Bronze is immutable raw store
# =============================================================================

import pandas as pd
import numpy as np

BRONZE_SCHEMA = StructType([
    StructField("ingestion_time", TimestampType(), False),
    StructField("event_time",     TimestampType(), False),
    StructField("region",         StringType(),    False),
    StructField("price_eur_mwh",  DoubleType(),    True),
    StructField("load_mw",        DoubleType(),    True),
    StructField("temperature_c",  DoubleType(),    True),
    StructField("source",         StringType(),    True),
])

def write_to_bronze_kql(records):
    """Write price records to Bronze KQL table."""

    if not records:
        print("⚠️  No records to write — skipping")
        return 0

    pdf = pd.DataFrame(records)

    # ------------------------------------------------------------------
    # Fix — Explicit type casting before createDataFrame
    # Prevents Arrow BufferHolder negative size error caused by
    # mixed None + numeric types in pandas columns
    # ------------------------------------------------------------------
    pdf["ingestion_time"] = pd.to_datetime(pdf["ingestion_time"], utc=True)
    pdf["event_time"]     = pd.to_datetime(pdf["event_time"],     utc=True)
    pdf["region"]         = pdf["region"].astype(str)
    pdf["source"]         = pdf["source"].astype(str)

    # Cast nullable numeric columns — replace None with np.nan
    # np.nan is float64 — Arrow handles homogeneous float64 correctly
    pdf["price_eur_mwh"] = pd.to_numeric(pdf["price_eur_mwh"], errors="coerce").astype("float64")
    pdf["load_mw"]       = pd.to_numeric(pdf["load_mw"],       errors="coerce").astype("float64")
    pdf["temperature_c"] = pd.to_numeric(pdf["temperature_c"], errors="coerce").astype("float64")

    # ------------------------------------------------------------------
    # Disable Arrow optimization for this createDataFrame call
    # Arrow fails on nullable float64 columns with NaN values
    # Spark falls back to row-by-row serialization — safe and correct
    # ------------------------------------------------------------------
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "false")

    df_spark = spark.createDataFrame(pdf, schema=BRONZE_SCHEMA)

    # Re-enable Arrow for rest of notebook operations
    spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")

    # Write to KQL Bronze — append only
    df_spark.write \
        .format("com.microsoft.kusto.spark.datasource") \
        .option("kustoCluster",  KUSTO_CLUSTER) \
        .option("kustoDatabase", KUSTO_DATABASE) \
        .option("kustoTable",    KUSTO_TABLE) \
        .option("accessToken",   mssparkutils.credentials.getToken("kusto")) \
        .mode("append") \
        .save()

    print(f"✅ Written {len(records)} records to {KUSTO_DATABASE}.{KUSTO_TABLE}")
    return len(records)

print("✅ Bronze KQL writer defined — Arrow fix applied")

StatementMeta(, 4f4dfe2d-dd38-4fee-8fb2-abc63e948b5f, 20, Finished, Available, Finished, False)

✅ Bronze KQL writer defined — Arrow fix applied


In [7]:
# =============================================================================
# CELL 6 — Main Execution
# =============================================================================
# Purpose : Orchestrate the full daily price poll cycle.
#           Scheduled via Fabric Pipeline once per day at 13:00 CET.
#
# Cycle summary:
#   1. Fetch day-ahead prices for all 26 ENTSO-E zones
#   2. Write to Bronze KQL table
#   3. Log summary
#
# Daily API calls: 26 (one per zone) — 0.002% of 400/min limit
# =============================================================================

cycle_start = datetime.now(timezone.utc)

print("=" * 55)
print("  PulseGrid — Daily Price Poller")
print(f"  Run time: {cycle_start.strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("=" * 55)

# Step 1 — Fetch prices
records = fetch_all_day_ahead_prices()

# Step 2 — Write to Bronze
written = write_to_bronze_kql(records)

# Step 3 — Summary
duration = (datetime.now(timezone.utc) - cycle_start).total_seconds()

print(f"\n{'='*55}")
print(f"  Daily Price Poller — Complete")
print(f"  Records written : {written}")
print(f"  Duration        : {duration:.1f}s")
print(f"  Next run        : Tomorrow 13:00 CET (via Fabric Pipeline)")
print(f"{'='*55}")

StatementMeta(, 4f4dfe2d-dd38-4fee-8fb2-abc63e948b5f, 21, Finished, Available, Finished, False)

  PulseGrid — Daily Price Poller
  Run time: 2026-08-14 06:41:17 UTC

📡 Fetching day-ahead prices for 2026-08-14
   Zones to poll: 27
   ✅ FR: 96 hourly records
   ✅ ES: 96 hourly records
   ✅ NL: 96 hourly records
   ✅ BE: 96 hourly records
   ✅ PL: 96 hourly records
   ✅ AT: 96 hourly records
   ✅ CH: 24 hourly records
   ✅ PT: 96 hourly records
   ✅ FI: 96 hourly records
   ✅ CZ: 96 hourly records
   ✅ SK: 96 hourly records
   ✅ HU: 96 hourly records
   ✅ RO: 96 hourly records
   ✅ BG: 96 hourly records
   ✅ HR: 96 hourly records
   ✅ GR: 96 hourly records
   ✅ SI: 96 hourly records
   ✅ RS: 96 hourly records
   ✅ LT: 96 hourly records
   ✅ LV: 96 hourly records
   ✅ DE-LU: 96 hourly records
   ✅ IT-NO: 96 hourly records
   ✅ DK-1: 96 hourly records
   ✅ DK-2: 96 hourly records
   ✅ SE-3: 96 hourly records
   ✅ NO-2: 96 hourly records
   ✅ EE: 96 hourly records

   Total records  : 2520
   Failed regions : None
✅ Written 2520 records to pulsegrid_bronze.raw_electricity_prices

  Dai